# Demonstration of Luma Module 
This notebook contain the implementation of the source code for each module in the EPISTEM land cover mapping framework

### Library import and earth engine initialization
If you have earth engine account you could used that to authenticate and initialize the earth engine. However, if you did not have the account, service account initialization is avaliable

In [ ]:
#This code is used if the notebook is implemented in github codespace. Just remove the (#)
!python -m pip install .. --quiet

In [ ]:
import ee 
import luma_ge

#Option 1: Manual authenticate using personal account
#Instructions for manual authentication
luma_ge.print_auth_instructions()
#uncomment the below line and follow earth engine authentication process
luma_ge.authenticate_manually()

#Option 2: Autheticate using service account (json file)
#service_account_path = '../auth/earth-engine-451407-520c1ef64879.json' #(if failed use this)
service_account_path = '../auth/ee-epstm2024.json'
success = luma_ge.initialize_with_service_account(service_account_path)

if success:
    print("Earth Engine initialized with service account successfully!")
else:
    print("Service account initialization failed. Try to authenticate earth engine manually")

#Check authentication status
status = luma_ge.get_auth_status()
print(f"Initialized: {status['initialized']}")
print(f"Authenticated: {status['authenticated']}")
if status['project']:
    print(f"Project: {status['project']}")

# Module 1: Acquisition of Near-Cloud-Free Satellite Imagery

### Shapefile Validation Testing

In [ ]:
# Import the validators
from luma_ge.input_utils import shapefile_validator, kml_validator
import geopandas as gpd

In [ ]:
# #Initialize shapefile validator
# print("=" * 60)
# print("SHAPEFILE VALIDATION TEST")
# print("=" * 60)
# #initilizae the validator
# validator_shp = shapefile_validator(verbose=True)
# #Path to test shapefile. can be change accordingly
# shapefile_path = '../data/area_of_interest.shp'
# #for demo, used geopandas to load the shapefile
# test_gdf = gpd.read_file(shapefile_path)
# print(f"\nLoaded shapefile with {len(test_gdf)} features")
# print(f"Geometry types: {test_gdf.geometry.geom_type.unique()}")

# #Validate and fix geometry
# print("\n--- Starting Validation ---")
# validated_gdf = validator_shp.validate_and_fix_geometry(test_gdf, geometry="mixed")

# if validated_gdf is not None:
#     print(f"\n✓ Validation successful!")
#     print(f"Features after validation: {len(validated_gdf)}")
#     print(f"All geometries valid: {validated_gdf.geometry.is_valid.all()}")
# else:
#     print(f"\n✗ Validation failed!")

In [ ]:
# Initialize KML validator
# print("\n" + "=" * 60)
# print("KML/KMZ VALIDATION TEST")
# print("=" * 60)
# #initilizate the class
# validator_kml = kml_validator(verbose=True)
# kml_path = '../data/AOI_Kab_PagarAlam.kmz'  #Can be change accordingly
# #load the data
# try:
#     #perform the validation
#     validated_kml_gdf = validator_kml.load_and_validate(kml_path, geometry="mixed")
    
#     if validated_kml_gdf is not None:
#         print(f"\n✓ KML validation successful!")
#         print(f"Features after validation: {len(validated_kml_gdf)}")
#         print(f"CRS: {validated_kml_gdf.crs}")
#         print(f"All geometries valid: {validated_kml_gdf.geometry.is_valid.all()}")
#     else:
#         print(f"\n✗ KML validation failed!")
        
# except FileNotFoundError:
#     print(f"⚠ KML file not found at {kml_path}")
#     print("To test KML validation, provide a valid KML file path")
# except Exception as e:
#     print(f"⚠ Error testing KML: {e}")

### System Response 1.1: Area of Interest Definition

In [ ]:
import geemap
from luma_ge.data_acquisition import Reflectance_Data, Reflectance_Stats, final_Image
from luma_ge.helpers import get_aoi_from_gaul

In [ ]:
#Set the country and province for the AOI using GAUL admin boundaries
#aoi = get_aoi_from_gaul(country="Indonesia", province="Sumatera Selatan")
#Alternatively, used geemap_shp_to_ee to directly used shapefile in your local machine

In [ ]:
#alternatively, you can also select smaller AOI using regency data below
import pandas as pd
indo_regency = ee.FeatureCollection('projects/ee-agilakbar/assets/Indonesian_Regency')
#check Regency List
regency = indo_regency.aggregate_array("WADMKK").getInfo()
#print(pd.DataFrame(regency, columns=["WADMKK"]))
#province = indo_regency.aggregate_array("WADMPR").getInfo()
#Example for Pagar Alam
regency_name = "Kota Pagar Alam"
#Filter the FeatureCollection, used it for AOI
aoi = indo_regency.filter(ee.Filter.eq("WADMKK", regency_name)).geometry()

### System Response 1.2: Search and Filter Imagery
The EPISTEM source code supports Landsat mission data, ranging from Landsat 1 to Landsat 9. For Landsat 1 - 3, the avaliable data is corrected radiance reflectance. The Landsat 5-9 used here is collection 2 surface reflectance (SR) analysis ready data.

The retrival logic used here is as follow:
1. Retrive multispectral bands (band 1 - 7) from landsat collection 2 SR data (if avaliable)
2. Retrive thermal band from landsat collection 2 TOA data 
3. Create temporal composite for each data 
4. Stacked the final two data into a earth engine image (ee.image)

In [ ]:
#========== FIRST RETRIVE THE MULTISPECTRAL BAND===========
#Intialize the relfectance class data function
optical_reflectance = Reflectance_Data()
#Initialize the final image class for composite creation
composite = final_Image() #NEW FEATURE ADDED HERE
#define the start and end date for imagery collection
start = '2025-01-01'
end = '2025-12-31'
#get the image collection and corresponding statistics
landsat_data, meta = optical_reflectance.get_optical_data(aoi, start, end, optical_data='L9_SR', 
                                                           cloud_cover=40, compute_detailed_stats=False)
#create mosaic between image collection, and clip based on AOI
mosaic_landsat = composite.get_quality_mosaic(landsat_data, aoi, quality_band= 'NDVI', calculate_coverage=False) #REPLACE OLD CODE WITH THE NEW ONE HERE
#Alternatively you can use temporal aggregation (ee reducer) to create mode cloudless imagery
#Add new functionality to calculate the coverage of the composite
median_landsat, coverage = composite.get_temporal_composite(landsat_data, aoi, reducer='Median', calculate_coverage=True) #REPLACE OLD CODE WITH THE NEW ONE HERE
#visualization parameter
l8_sr_visparam = {'min': 0,'max': 0.4,'gamma': [0.95, 1.1, 1],'bands':['NIR', 'RED', 'GREEN']}
#Add the data to the map
Map = geemap.Map()
Map.addLayer(mosaic_landsat, l8_sr_visparam, 'L8 SR Mosaic')
Map.addLayer(median_landsat, l8_sr_visparam, 'L8 SR Median')
Map.addLayer(landsat_data, l8_sr_visparam, 'L8 SR Image Collection')
# set center of the map in the area of interest
Map.centerObject(aoi, 7)

In [ ]:
#retive thermal bands from TOA
thermal_bands, thermal_stats = optical_reflectance.get_thermal_bands(aoi, start, end, cloud_cover=40, thermal_data='L9_TOA', compute_detailed_stats=False)
median_thermal = composite.get_temporal_composite(thermal_bands, aoi, reducer='Median') #REPLACE THE OLD CODE WITH THE NEW ONE
thermal_vis = {'min': 286,'max': 300,'gammma': 0.4}
#stacked all landsat bands and convert to float(making sure all data type are compatible)
stacked_landsat = median_landsat.addBands(median_thermal).toFloat()
#visualize the thermal bands and multispectral bands
Map.addLayer(median_thermal, thermal_vis, "Thermal Bands")
Map

### Image retrival report (optional)

In [ ]:
# #intialize the statistic class
# stats = Reflectance_Stats()
# #get the retrival report and automatically print them
# retrival_report = stats.get_collection_statistics(landsat_data, print_report=True)

### System Response 1.3: Imagery Download

In [ ]:
# export_task = ee.batch.Export.image.toDrive(
#     image=stacked_landsat,
#     description='Landsat_Median_composite_2017_Sumsel',
#     folder='Earth Engine',
#     fileNamePrefix='Landsat_Median_composite_2017_Sumsel',
#     scale=30,
#     region=aoi,  # or aoi.geometry()
#     maxPixels=1e13
# )
# export_task.start()
# import time

# while export_task.active():
#     print('Exporting... (status: {})'.format(export_task.status()['state']))
#     time.sleep(10)

# print('Export complete (status: {})'.format(export_task.status()['state']))

# Module 2:  Land-cover classification Scheme
Three approach are provided to handle classification scheme:
1. Upload a csv file 
2. Manual input the classification scheme
3. Use default classification scheme (RESTORE+ project)

### Import the module

In [8]:
from luma_ge.classification_scheme import LULC_Scheme_Manager
#Initialize the LULC Scheme Manager
manager = LULC_Scheme_Manager()
print("Land Cover Classification Scheme Manager initialized!")
print(f"Current class count: {manager.get_class_count()}")
#Temporary function to display the classiifcation scheme in notebook
#Display current classification scheme
def display_classification_scheme(manager):
    """Display the current classification scheme in a readable format"""
    if not manager.has_classes():
        print("No classes defined yet.")
        return
    
    print("\n=== Current Classification Scheme ===")
    df = manager.get_dataframe()
    print(df.to_string(index=False))
    
    return df

# Display the scheme
df = display_classification_scheme(manager)

Land Cover Classification Scheme Manager initialized!
Current class count: 0
No classes defined yet.


### System Response 2.1a: Upload Classification Scheme

In [9]:
# import pandas as pd
# #Reset manager for CSV upload example
# manager = LULC_Scheme_Manager()
# #path to csv 
# csv_path = "../data/Example_Classification_scheme.csv"

# print("=== CSV Upload Process ===")

# # Load the CSV
# df = pd.read_csv(csv_path, sep=None, engine="python")
# print("Loaded CSV:")
# print(df)

# # Auto-detect columns
# id_col, name_col, color_col = manager.auto_detect_csv_columns(df)
# print(f"\nAuto-detected columns:")
# print(f"ID column: {id_col}")
# print(f"Name column: {name_col}")
# print(f"Color column: {color_col}")

=== CSV Upload Process ===
Loaded CSV:
   ID Land Cover Class Color Palette
0   1            Karet       #5a9a67
1   2     Kelapa Sawit       #bbbb5a
2   3        Tubuh Air       #111eda
3   4       Permukiman       #da0407

Auto-detected columns:
ID column: ID
Name column: Land Cover Class
Color column: Color Palette


In [10]:
# # Process CSV upload
# success, message = manager.process_csv_upload(df, id_col, name_col, color_col)
# if success:
#     print(f"✅ {message}")
    
#     # Finalize the upload
#     success, message = manager.finalize_csv_upload()
#     if success:
#         print(f"✅ {message}")
#     else:
#         print(f"❌ {message}")
# else:
#     print(f"❌ {message}")

# # Display the loaded scheme
# display_classification_scheme(manager)

✅ Successfully loaded 4 classes from CSV with colors from CSV
✅ Skema klasifikasi berhasil dibuat dengan 4 kelas

=== Current Classification Scheme ===
 ID Land Cover Class Color Palette
  1            Karet       #5a9a67
  2     Kelapa Sawit       #bbbb5a
  3        Tubuh Air       #111eda
  4       Permukiman       #da0407


,ID,Land Cover Class,Color Palette
0,1,Karet,#5a9a67
1,2,Kelapa Sawit,#bbbb5a
2,3,Tubuh Air,#111eda
3,4,Permukiman,#da0407


### System Response 2.1b: Manual Scheme Definition

In [ ]:
# #Reset manager for manual input example
# manager = LULC_Scheme_Manager()
# #Manually add the class
# print("=== Manual Class Addition ===")

# #Example of class to add
# classes_to_add = [
#     (1, "Hutan Lahan Kering", "#0E6D0E"),
#     (2, "Pertanian Lahan Kering", "#E8F800"),
#     (3, "Permukiman", "#F81D00"),
#     (4, "Badan Air", "#1512F3"),
#     (5, "Pertanian Lahan Basah", "#")
# ]

# for class_id, class_name, color_code in classes_to_add:
#     success, message = manager.add_class(class_id, class_name, color_code)
#     if success:
#         print(f"✅ {message}")
#     else:
#         print(f"❌ {message}")

# print(f"\nTotal classes: {manager.get_class_count()}")

In [ ]:
# Example: Edit an existing class
print("=== Editing a Class ===")

# Edit the first class (index 0)
class_to_edit = manager.edit_class(0)
if class_to_edit:
    print(f"Editing class: {class_to_edit}")
    
    # Update the class with new information
    success, message = manager.add_class(1, "HUtan Lahan Rendah", "#004D00")
    if success:
        print(f"✅ {message}")
    else:
        print(f"❌ {message}")

# Display updated scheme
display_classification_scheme(manager)

### System Response 2.1c: Template Classification Scheme

In [11]:
# Reset manager for default scheme example
manager = LULC_Scheme_Manager()

print("=== Available Default Schemes ===")
default_schemes = manager.get_default_schemes()

for scheme_name, classes in default_schemes.items():
    print(f"\n{scheme_name}: {len(classes)} classes")
    for class_data in classes:
        print(f"  - ID {class_data['ID']}: {class_data['Class Name']} ({class_data['Color Code']})")

=== Available Default Schemes ===

RESTORE+ Project: 17 classes
  - ID 1: Undisturbed dry-land forest (#006400)
  - ID 2: Logged-over dry-land forest (#228B22)
  - ID 3: Undisturbed mangrove (#4169E1)
  - ID 4: Logged-over mangrove (#87CEEB)
  - ID 5: Undisturbed swamp forest (#2E8B57)
  - ID 6: Logged-over swamp forest (#8FBC8F)
  - ID 7: Agroforestry (#9ACD32)
  - ID 8: Plantation forest (#32CD32)
  - ID 9: Rubber monoculture (#8B4513)
  - ID 10: Oil palm monoculture (#FF8C00)
  - ID 11: Other monoculture (#DAA520)
  - ID 12: Grass/savanna (#ADFF2F)
  - ID 13: Shrub (#90EE90)
  - ID 14: Cropland (#FFFF00)
  - ID 15: Settlement (#FF0000)
  - ID 16: Cleared land (#D2B48C)
  - ID 17: Waterbody (#0000FF)


In [ ]:
# Load the RESTORE+ default scheme
scheme_name = "RESTORE+ Project"
success, message = manager.load_default_scheme(scheme_name)

if success:
    print(f"✅ {message}")
else:
    print(f"❌ {message}")

# Display the loaded scheme
display_classification_scheme(manager)

#### Select class of interest

In [ ]:
manager = LULC_Scheme_Manager()
manager.load_default_scheme("RESTORE+ Project")
df = manager.get_dataframe()  # <- this provides the classification_df

selection = manager.store_classes_of_interest(
    scheme_name="RESTORE+ Project",
    classes_of_interest=[1]
)

print(selection)

### System Response 2.2: Download classification scheme

In [12]:
print("=== Export Classification Scheme ===")
#Convert the selected  classification scheme manager to dataframe
classification_df = manager.get_dataframe()
print("Classification DataFrame:")
print(classification_df)
#Save the file
output_path = '../Selected_LC_Classification_Scheme.csv'
classification_df.to_csv(output_path, index=False)
print(f"\n✅ Classification scheme saved to: {output_path}")

=== Export Classification Scheme ===
Classification DataFrame:
   ID Land Cover Class Color Palette
0   1            Karet       #5a9a67
1   2     Kelapa Sawit       #bbbb5a
2   3        Tubuh Air       #111eda
3   4       Permukiman       #da0407

✅ Classification scheme saved to: ../Selected_LC_Classification_Scheme.csv


## Module 3: Generate Region Of Interest
Three methods to generate ROI are supported in EPISTEM platform:
1. **Upload Training Data** - Upload your own shapefile
2. **On-screen Sampling** - Create samples using interactive map
3. **Default Reference Data** - Use Epistem's default training data

### Library Import and Setup

### System Response 3.1 Prerequisite Check

In [13]:
print("=== Checking Prerequisites ===")
#Load from previous module
#From Module 1 - AOI data
try:
    AOI = aoi
    print("✅ AOI from Module 1 is available")
    aoi_available = True
except:
    print("❌ AOI data not available, please run Module 1 first")
    aoi_available = False

#From Module 2 - Classification scheme
try:
    
    # For demonstration, create sample classification scheme
    LULCTable = classification_df
    print("✅ Classification scheme from Module 2 is available")
    print(f"   - Number of classes: {len(LULCTable)}")
    scheme_available = True
except:
    print("❌ Classification scheme not available, please run Module 2 first")
    scheme_available = False

if aoi_available and scheme_available:
    print("\n✅ All prerequisites met! You can proceed with training data collection.")
else:
    print("\n❌ Prerequisites not met. Please complete previous modules first.")

=== Checking Prerequisites ===
✅ AOI from Module 1 is available
✅ Classification scheme from Module 2 is available
   - Number of classes: 4

✅ All prerequisites met! You can proceed with training data collection.


In [14]:
# Modul 3a 
# Import modules and functions
import pandas as pd
from luma_ge.sample_data import SyncTrainData

## System Response 3.2 ROI Upload and content Verification

In [15]:
# # ----- Data Input -----
# # 1. Decision to upload data
# UploadTrainData = True # set as 'true' to upload your own training data shapefile
# # set as 'false' to either add train data by sampling on screen or use default training data

# # 2. Training data file path (if UploadTrainData is true)
# TrainVectPath  = '../data/Training_Sumsel_Data.shp'
# TrainField = 'ID' 
#         # Load and process training data
# TrainDataDict = SyncTrainData.LoadTrainData(
#             landcover_df=LULCTable,
#             aoi_geometry=AOI,
#             training_shp_path=TrainVectPath
#         )

2026-02-26 11:30:01,056 - luma_ge.sample_data - INFO - Loading training data from shapefile: ../data/training_points.shp
2026-02-26 11:30:01,087 - luma_ge.sample_data - WARNING - 'kelas' field not found in training data
2026-02-26 11:30:01,087 - luma_ge.sample_data - INFO - Available columns: ['LULC_Type', 'ID', 'geometry']


In [16]:
# # ----- System response 3.2.a -----
# # Set class field
# TrainDataDict = SyncTrainData.SetClassField(TrainDataDict, TrainField)

# # Validate classes
# TrainDataDict = SyncTrainData.ValidClass(TrainDataDict, 1)

#     # Check sample sufficiency
# TrainDataDict = SyncTrainData.CheckSufficiency(TrainDataDict, min_samples=20)

#     # Filter by AOI
# TrainDataDict = SyncTrainData.FilterTrainAoi(TrainDataDict)

#     # Create training data table
# table_df, total_samples, insufficient_df = SyncTrainData.TrainDataRaw(
#     training_data=TrainDataDict.get('training_data'),
#     landcover_df=TrainDataDict.get('landcover_df'),
#     class_field=TrainDataDict.get('class_field'))

# #Summary result
# vr = TrainDataDict.get('validation_results', {})

# print("=" * 70)
# print("TRAINING DATA SUMMARY")
# print("=" * 70)
# print(f"Total training points loaded     : {vr.get('total_points', 'N/A')}")
# print(f"Points after class filtering     : {vr.get('points_after_class_filter', 'N/A')}")
# print(f"Valid points (inside AOI)        : {vr.get('valid_points', 'N/A')}")
# print(f"Invalid classes found            : {len(vr.get('invalid_classes', []))}")
# print(f"Points outside AOI               : {len(vr.get('outside_aoi', []))}")
# print("=" * 70)

#     # --- Display the main table ---
# if table_df is not None and not table_df.empty:
#         display_df = table_df.copy()
#         if 'Percentage' in display_df.columns:
#             display_df['Percentage'] = display_df['Percentage'].apply(
#                 lambda x: f"{x:.2f}%" if isinstance(x, (int, float)) else x
#             )
#         display(display_df)
# else:
#         print("No valid training data available to display.")

### System Response 3.2 Default Training Data (RESTORE+)

In [ ]:
print(" Loading default reference training data...")
TrainEePath = 'projects/ee-rg2icraf/assets/Indonesia_lulc_Sample'
TrainField = 'kelas'

# Stopgap solution: Rename 'Land Cover Class' column to 'LULC_Type' if it exists
if 'Land Cover Class' in LULCTable.columns:
    LULCTable = LULCTable.rename(columns={'Land Cover Class': 'LULC_Type'})
    print("Column 'Land Cover Class' renamed to 'LULC_Type'")

    
try:
    print("Loading reference training data from Earth Engine...")
        
        # Load training data
    TrainDataDict = SyncTrainData.LoadTrainData(
            landcover_df=LULCTable,
            aoi_geometry=AOI,
            training_ee_path=TrainEePath
        )
        
    print("Processing and validating reference data...")
        
        # Set class field
    TrainDataDict = SyncTrainData.SetClassField(TrainDataDict, TrainField)
        
        # Validate classes
    TrainDataDict = SyncTrainData.ValidClass(TrainDataDict, use_class_ids=True)
        
        # Check sufficiency
    TrainDataDict = SyncTrainData.CheckSufficiency(TrainDataDict, min_samples=20)
        
        # Filter by AOI
    TrainDataDict = SyncTrainData.FilterTrainAoi(TrainDataDict)
        
        # Create summary table
    table_df, total_samples, insufficient_df = SyncTrainData.TrainDataRaw(
            training_data=TrainDataDict.get('training_data'),
            landcover_df=TrainDataDict.get('landcover_df'),
            class_field=TrainDataDict.get('class_field')
        )
        
    print("✅ Reference training data loaded and processed successfully!")
    print(f"Total samples: {total_samples}")
        
        # Display summary table
    display(table_df)
        
        # Store final training data
    TrainDataFinal = TrainDataDict.get('training_data')
        
        # Show validation results
    vr = TrainDataDict.get('validation_results', {})
    print(f"\nValidation Results:")
    print(f"- Total points loaded: {vr.get('total_points', 'N/A')}")
    print(f"- Points after class filter: {vr.get('points_after_class_filter', 'N/A')}")
    print(f"- Valid points (within AOI): {vr.get('valid_points', 'N/A')}")
    print(f"- Invalid classes: {len(vr.get('invalid_classes', []))}")
        
except Exception as e:
        print(f"❌ Error loading reference data: {e}")
        TrainDataFinal = None

## Module 4: Region of Interest Separability Analysis

### Library Import and Setup

In [23]:
#Import the sample quality functions
from luma_ge.sample_data_quality import sample_quality, spectral_plotter
#if there's error in the import, uncomment the below line to install the PyCRS package
#!pip install PyCRS

### System Response 4.1 Computing Separability Analysis

In [24]:
# roi_path = '../data/Training_Sumsel_Data.shp'
# labeled_roi = geemap.shp_to_ee('../data/Training_Sumsel_Data.shp')
# # labeled_roi = geemap.gdf_to_ee(TrainDataFinal)

# #Conduct the analysis
# analyzer = sample_quality(training_data=labeled_roi, 
#     image= stacked_landsat, 
#     class_property='ID',           # Column with numeric IDs (1, 2, 3, etc.)
#     region= aoi,
#     class_name_property='LC_Name'          # Column with names ('Forest', 'Urban', 'Water', etc.)
# )
# # Extract spectral values
# pixel_extract = analyzer.extract_spectral_values(scale=100, max_pixels_per_class=5000)
# samples_statistic = analyzer.sample_stats()
# sample_df = analyzer.get_sample_stats_df()
# display(sample_df)
# #Sample statistic

Extracted spectral values for 44 samples across 4 classes


,ID,LULC_Type,Sample_Count,Proportion,Percentage
0,1,Karet,10,0.2273,22.73
1,2,Kelapa Sawit,10,0.2273,22.73
2,3,Tubuh Air,12,0.2727,27.27
3,4,Permukiman,12,0.2727,27.27


In [25]:
# #Sample statistic (pixel value extracted from the imagery)
# pixel_stats = analyzer.sample_pixel_stats(pixel_extract)
# pixel_stats_df = analyzer.get_sample_pixel_stats_df(pixel_extract)
# display(pixel_stats_df)

,ID,LULC_Type,Band,Mean,Std,Min,Max,Median,Count
0,1,Karet,AEROSOL,0.03,0.01,0.02,0.04,0.03,10
1,1,Karet,BLUE,0.03,0.01,0.02,0.04,0.03,10
2,1,Karet,GREEN,0.05,0.01,0.04,0.07,0.05,10
3,1,Karet,NIR,0.36,0.03,0.29,0.41,0.37,10
4,1,Karet,RED,0.04,0.01,0.03,0.06,0.03,10
5,1,Karet,SWIR1,0.19,0.02,0.15,0.22,0.19,10
6,1,Karet,SWIR2,0.08,0.01,0.06,0.11,0.08,10
7,1,Karet,THERMAL,298.57,1.70,296.20,300.95,297.72,10
8,2,Kelapa Sawit,AEROSOL,0.03,0.01,0.02,0.04,0.03,10
9,2,Kelapa Sawit,BLUE,0.03,0.01,0.02,0.05,0.03,10


In [26]:
# #Perform separability Analysis (iether using Transformed Divergence or Jeffries Matutista )
# separability_analysis = analyzer.get_separability_df(pixel_extract, method='TD')
# display(separability_analysis)

In [ ]:
# #Get the lowest separability
# lowest_sep = analyzer.lowest_separability(pixel_extract)
# display(lowest_sep)

In [ ]:
# # Overall separability summary
# sep_summary = analyzer.sum_separability(pixel_extract)
# print("Overall Separability Statistics:")
# display(sep_summary)

## System Response 4.2 Sample Visualization

In [ ]:
# #Box plot to detect outlier
# ploter = spectral_plotter(analyzer)
# box = ploter.plot_boxplot(pixel_extract)
# for fig in box:
#     fig.show()

In [17]:
# #static scatter plot
# stat_plot = ploter.static_scatter_plot(pixel_extract, x_band='NIR', y_band='RED', add_ellipse=True)

2026-02-26 11:30:13,930 - luma_ge.predictor - INFO - Terrain calculator initialized
2026-02-26 11:30:13,931 - luma_ge.predictor - INFO - Calculating elevation layer using NASADEM DEM...
2026-02-26 11:30:13,932 - luma_ge.predictor - INFO - Successfully calculated elevation layer using NASADEM DEM
2026-02-26 11:30:13,932 - luma_ge.predictor - INFO - Calculating slope layer using NASADEM DEM...
2026-02-26 11:30:13,933 - luma_ge.predictor - INFO - Calculating elevation layer using NASADEM DEM...
2026-02-26 11:30:13,933 - luma_ge.predictor - INFO - Successfully calculated elevation layer using NASADEM DEM
2026-02-26 11:30:13,934 - luma_ge.predictor - INFO - Successfully calculated slope layer using NASADEM DEM
2026-02-26 11:30:13,934 - luma_ge.predictor - INFO - Calculating aspect layer using NASADEM DEM...
2026-02-26 11:30:13,935 - luma_ge.predictor - INFO - Calculating elevation layer using NASADEM DEM...
2026-02-26 11:30:13,935 - luma_ge.predictor - INFO - Successfully calculated elevati

In [ ]:
# #3D scatter plot
# multi_d_scater = ploter.scatter_plot_3d(pixel_extract)
# multi_d_scater

2026-02-26 11:30:50,117 - luma_ge.predictor - INFO - Correlation analysis initialized with spearman method
2026-02-26 11:30:50,118 - luma_ge.predictor - INFO - Computing spearman correlation matrix with 5000 samples at 30m scale
2026-02-26 11:30:50,118 - luma_ge.predictor - INFO - Generating 5000 random sample points with seed 42
2026-02-26 11:30:50,119 - luma_ge.predictor - INFO - Successfully generated 5000 random sample points


Computing correlation matrix...


2026-02-26 11:30:50,475 - luma_ge.predictor - INFO - Processing 20 bands: ['AEROSOL', 'BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2', 'THERMAL', 'elevation', 'slope', 'aspect', 'NDVI', 'GBNDVI', 'MSAVI', 'NDMI', 'EVI', 'MBI', 'NSDS', 'MNDWI', 'AWEIsh']


## Module 6: Land Cover Classification

### Library Import

In [ ]:
from luma_ge.classification import FeatureExtraction, Generate_LULC
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### System Response 6.2 Classification

In [ ]:
labeled_roi = geemap.gdf_to_ee(TrainDataFinal)

#Perform Training Test Split
features = FeatureExtraction()
strafied_train, stratified_test = features.stratified_split(labeled_roi, stacked_landsat, 
                            class_prop='kelas', train_ratio=0.5)


In [ ]:
classifier = Generate_LULC()
print("Performing Classification...")
#Multiclass hard classification
classification_map, trained_model = classifier.hard_classification(strafied_train, class_property='kelas', image=stacked_landsat,
                                                          ntrees=300, min_leaf=2, return_model=True)
# Evaluate model performance
print("Evaluating model performance...")

try:
    accuracy_metrics = classifier.evaluate_model(
        trained_model=trained_model,
        test_data=stratified_test,
        class_property='kelas'
    )
    
    print("✓ Model evaluation completed")
    
except Exception as e:
    print(f"❌ Error in model evaluation: {e}")                                                    


In [ ]:
orig_hist = classification_map.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=aoi,
    scale=30,
    maxPixels=1e13
).getInfo()

print(orig_hist)

### System Response 6.3 Model Evaluation

In [ ]:
# # Display accuracy results
# print("=== Model Performance Summary ===")
# print(f"Overall Accuracy: {accuracy_metrics['overall_accuracy']:.4f} ({accuracy_metrics['overall_accuracy']*100:.2f}%)")
# print(f"Kappa Coefficient: {accuracy_metrics['kappa']:.4f}")
# print(f"Overall G-Mean: {accuracy_metrics['overall_gmean']:.4f}")

# print("\n=== Per-Class Metrics ===")
# #Class Dataframe
# metrics_df = pd.DataFrame({
#     'Precision': accuracy_metrics['precision'],
#     'Recall': accuracy_metrics['recall'],
#     'F1-Score': accuracy_metrics['f1_scores'],
#     'G-Mean': accuracy_metrics['gmean_per_class']
# })

# # Round to 4 decimal places
# metrics_df = metrics_df.round(4)

# display(metrics_df)

In [ ]:
# # Visualize confusion matrix
# confusion_matrix = np.array(accuracy_metrics['confusion_matrix'])
# plt.figure(figsize=(8, 6))
# sns.heatmap(confusion_matrix, 
#             annot=True, 
#             fmt='d', 
#             cmap='Blues')
# plt.title('Confusion Matrix')
# plt.xlabel('Predicted Class')
# plt.ylabel('Actual Class')
# plt.tight_layout()
# plt.show()

In [ ]:
# # Get feature importance
# print("Analyzing feature importance...")

# try:
#     importance_df = classifier.get_feature_importance(trained_model)
#     print("✓ Feature importance analysis completed")
    
#     display(importance_df)
    
# except Exception as e:
#     print(f"❌ Error in feature importance analysis: {e}")
# # Visualize feature importance
# plt.figure(figsize=(10, 6))

# # Create bar plot
# bars = plt.bar(importance_df['Band'], importance_df['Importance'])

# # Add value labels on bars
# for bar, value in zip(bars, importance_df['Importance']):
#     plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
#              f'{value:.1f}%', ha='center', va='bottom')

# plt.title('Feature Importance by Spectral Band')
# plt.xlabel('Landsat 8 Bands')
# plt.ylabel('Importance')
# plt.xticks(rotation=45)
# plt.grid(axis='y', alpha=0.3)
# plt.tight_layout()
# plt.show()

## Classification Map Visualization

In [ ]:
# === Load Classification Scheme ===
scheme = pd.read_csv("../Selected_LC_Classification_Scheme.csv", sep=None, engine="python")
classes = [str(x).strip() for x in scheme["Land Cover Class"].tolist()]
palette = [str(x).strip() for x in scheme["Color Palette"].tolist()]
ids = scheme["ID"].tolist()
legend_dict = dict(zip(classes, palette))

# === Visualization Parameters ===
vis_params = {
    "min": min(ids),
    "max": max(ids),
    "palette": palette
}

# === Create geemap Map ===
Map = geemap.Map() 
Map.centerObject(aoi, 7)
Map.addLayer(classification_map, vis_params, "LULC Classification")

# # === Add Legend ===
Map.add_legend(
    title="Land Cover Classification", 
    legend_dict=legend_dict
    )

# Display
Map


### Map reclassification of default classification scheme

In [ ]:
# Reclassify the map

final_map, info = classifier.reclassify_map_by_classes(
    classification_map=classification_map,
    classification_df=df,
    selected_classes=selection
)

info

In [ ]:
# Prepare Reclassified Visualization


classes_of_interest = selection["classes_of_interest"]
other_class_id = 999

scheme_filtered = scheme[scheme["ID"].isin(classes_of_interest)]

reclass_names = scheme_filtered["Land Cover Class"].tolist()
reclass_colors = scheme_filtered["Color Palette"].tolist()
reclass_ids = scheme_filtered["ID"].tolist()

# Add "Other"
reclass_names.append("Other")
reclass_colors.append("#BDBDBD")
reclass_ids.append(other_class_id)

# Sequential IDs for visualization
vis_ids = list(range(1, len(reclass_ids) + 1))

# Remap for visualization
vis_map = final_map.remap(reclass_ids, vis_ids)

reclass_vis_params = {
    "min": 1,
    "max": len(vis_ids),
    "palette": reclass_colors
}


# Add Reclassified Layer

Map.addLayer(
    vis_map,
    reclass_vis_params,
    "Reclassified LULC"
)

reclass_legend = dict(zip(reclass_names, reclass_colors))

Map.add_legend(
    title="Reclassified Land Cover",
    legend_dict=reclass_legend
)

Map

## Module 7: Thematic Accuracy Assessment

### System Response 7.3 Thematic Accuracy Assessment

In [ ]:
# from luma_ge.accuracy import Thematic_Accuracy_Assessment
# #Initialize the accuracy assessment class
# accuracy_assessor = Thematic_Accuracy_Assessment()
# print("✓ Thematic Accuracy Assessment class initialized")
# print(f"Supported metrics: {accuracy_assessor.supported_metrics}")

In [ ]:
# validation_data = geemap.shp_to_ee("../data/Evaluation_Sumsel_data.shp") 

# # === 2. Create Assessment Object ===
# assessor = Thematic_Accuracy_Assessment()

# # === 3. Run Accuracy Assessment ===
# success, results = assessor.run_accuracy_assessment(
#     lcmap=classification_map,
#     validation_data=validation_data,
#     class_property='LULC_ID',   #Validation ID column
#     scale=30
# )

# # === 4. Display Results ===
# if success:
#     print("=== Thematic Accuracy Results ===")
#     summary = assessor.format_accuracy_summary(results)
#     print("Overall Accuracy :", summary['overall_accuracy'])
#     print("Kappa Coefficient:", summary['kappa'])
#     print("95% CI          :", summary['confidence_interval'])
#     print("Samples Used     :", summary['sample_size'])
# else:
#     print("Error:", results["error"])
